# Data Acquisition from Materials Databases

This notebook retrieves materials science datasets from external APIs and databases.

## Sources
- Materials Project API (requires API key)
- JARVIS Database
- Other materials science repositories

## Steps
1. Load API credentials from `.env`
2. Query materials databases
3. Save raw data to CSV files in `../data/`
4. Document data sources and acquisition parameters

In [5]:
# Load environment variables
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from pathlib import Path
from dotenv import load_dotenv
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Matminer — dataset loader and featurizers
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import StructureToComposition
from matminer.featurizers.base import MultipleFeaturizer
from pymatgen.core import Composition
import typing
from typing_extensions import NotRequired
typing.NotRequired = NotRequired

# Set up paths
PROJECT_ROOT = Path().cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

print('✅ All imports successful')

✅ All imports successful


In [12]:
# Query Materials Project for binary carbon compounds with transition metals
from mp_api.client import MPRester
import os

# Get API key from environment
api_key = os.getenv('MP_API_KEY')
if not api_key:
    raise ValueError("MP_API_KEY not found in .env file. Get one at https://materialsproject.org/api")

# Initialize Materials Project client
mpr = MPRester(api_key=api_key)
print("Connected to Materials Project API")

Connected to Materials Project API


In [25]:
import re

def is_binary_c_tm(elements_list, tm):
    """Check if compound has exactly 2 elements: C and the specified transition metal"""
    if len(elements_list) != 2:
        return False
    
    # Get element symbols as strings
    element_symbols = set([str(elem).replace('Element ', '') for elem in elements_list])
    
    # Should have exactly C and the transition metal
    return element_symbols == {tm, 'C'}

# Query for binary compounds
transition_metals = [
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "La", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg"
]

carbon_tm_binary = []

for tm in transition_metals:
    print(f"Querying {tm}-C binary compounds...")
    try:
        materials = mpr.summary.search(
            elements=[tm, "C"],
            fields=[
                "material_id", 
                "formula_pretty", 
                "elements",
                "energy_above_hull", 
                "volume", 
                "density",
                "band_gap",
                "is_stable",
                "is_metal"
            ]
        )
        
        binary_found = 0
        if materials:
            for mat in materials:
                # Only keep if it's truly binary (exactly 2 elements: C and TM)
                if is_binary_c_tm(mat.elements, tm):
                    compound_data = {
                        'material_id': mat.material_id,
                        'formula': mat.formula_pretty,
                        'transition_metal': tm,
                        'energy_above_hull': mat.energy_above_hull,
                        'volume': mat.volume,
                        'density': mat.density,
                        'band_gap': mat.band_gap,
                        'is_stable': mat.is_stable,
                        'is_metal': mat.is_metal,
                    }
                    carbon_tm_binary.append(compound_data)
                    binary_found += 1
            print(f"  Found {binary_found} binary compounds (out of {len(materials)} total)")
    except Exception as e:
        print(f"Error querying {tm}-C: {e}")

print(f"\nTotal BINARY compounds found: {len(carbon_tm_binary)}")

# Save the filtered dataset
df_binary = pd.DataFrame(carbon_tm_binary)
output_file = DATA_DIR / 'carbon_tm_binary_compounds.csv'
df_binary.to_csv(output_file, index=False)
print(f"Saved {len(df_binary)} binary compounds to {output_file}")

# Show summary
print("\nBinary Dataset Summary:")
print(f"Shape: {df_binary.shape}")
print(f"Columns: {list(df_binary.columns)}")
print("\nFirst 10 rows:")
print(df_binary.head(10))

Task was destroyed but it is pending!
task: <Task pending name='Task-276' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-277' coro=<_async_in_context.<locals>.preserve_context() running at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-277' coro=<_async_in_context.<locals>.preserve_context() running at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]>


Querying Sc-C binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/101 [00:00<?, ?it/s]

  Found 11 binary compounds (out of 101 total)
Querying Ti-C binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/139 [00:00<?, ?it/s]

...

  Found 4 binary compounds (out of 115 total)
Querying Au-C binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/66 [00:00<?, ?it/s]

  Found 4 binary compounds (out of 66 total)
Querying Hg-C binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/130 [00:00<?, ?it/s]

  Found 3 binary compounds (out of 130 total)

Total BINARY compounds found: 215
Saved 215 binary compounds to /Users/benbenmerk/Merkin_FinalProject/data/carbon_tm_binary_compounds.csv

Binary Dataset Summary:
Shape: (215, 9)
Columns: ['material_id', 'formula', 'transition_metal', 'energy_above_hull', 'volume', 'density', 'band_gap', 'is_stable', 'is_metal']

First 10 rows:
  material_id formula transition_metal  energy_above_hull      volume  \
0   mp-999203     ScC               Sc           0.600033   53.364864   
1   mp-999205     ScC               Sc           0.578838   62.137402   
2  mp-1009751     ScC               Sc           0.921636   33.385907   
3  mp-1009748     ScC               Sc           0.839815   23.864912   
4  mp-1219333    ScC2               Sc           0.569321   27.126498   
5    mp-10020     ScC               Sc           0.308115   25.720312   
6    mp-16296   Sc2C3               Sc           0.112614  228.543378   
7    mp-29941    Sc2C               Sc 

In [18]:
# Convert to DataFrame
df = pd.DataFrame(carbon_tm_compounds)

# Save to CSV
output_file = DATA_DIR / 'carbon_tm_binary_compounds.csv'
df.to_csv(output_file, index=False)
print(f"Saved {len(df)} compounds to {output_file}")

# Show summary
print("\nDataset Summary:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 10 rows:")
print(df.head(10))

Saved 5881 compounds to /Users/benbenmerk/Merkin_FinalProject/data/carbon_tm_binary_compounds.csv

Dataset Summary:
Shape: (5881, 8)
Columns: ['material_id', 'formula', 'energy_above_hull', 'volume', 'density', 'band_gap', 'is_stable', 'is_metal']

First 10 rows:
  material_id   formula  energy_above_hull      volume    density  band_gap  \
0   mp-772763  K2ScPCO7           0.000000  386.884213   2.387538    4.3677   
1   mp-972937    ScAlCO           0.015525   93.159547   3.563066    0.3695   
2   mp-999203       ScC           0.600033   53.364864   3.545227    0.0000   
3   mp-999205       ScC           0.578838   62.137402   3.044713    0.0000   
4  mp-1009751       ScC           0.921636   33.385907   2.833390    0.0000   
5  mp-1009748       ScC           0.839815   23.864912   3.963781    0.0000   
6  mp-1018635    ScPd3C           0.499213   74.745083   8.358262    0.0000   
7  mp-1018628    ScPt3C           0.654464   76.605450  13.921060    0.0000   
8  mp-1080113    ScFeC2  

## TODO: Add data acquisition code
- Connect to Materials Project API
- Query for materials with specific properties
- Handle pagination and rate limits
- Save to CSV